In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, input_file_name, regexp_extract
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Lab2_ETL_Postgres") \
    .config("spark.driver.memory", "2g") \
    .config("spark.jars", "/home/jovyan/jars/postgresql-42.7.1.jar") \
    .getOrCreate()

print(f" Spark: {spark.version}")

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("nullValue", "") \
    .csv("/home/jovyan/mock_data/MOCK_DATA_*.csv")

print(f" Прочитано строк: {df_raw.count()}, колонок: {len(df_raw.columns)}")
df_raw.printSchema()

 Spark: 3.5.0
 Прочитано строк: 20022, колонок: 50
root
 |-- id: string (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_age: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_postal_code: string (nullable = true)
 |-- customer_pet_type: string (nullable = true)
 |-- customer_pet_name: string (nullable = true)
 |-- customer_pet_breed: string (nullable = true)
 |-- seller_first_name: string (nullable = true)
 |-- seller_last_name: string (nullable = true)
 |-- seller_email: string (nullable = true)
 |-- seller_country: string (nullable = true)
 |-- seller_postal_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_price: string (nullable = true)
 |-- product_quantity: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- sal

In [2]:
# Проверим, сколько уникальных id
print(f"Всего строк: {df_raw.count()}")
print(f"Уникальных id: {df_raw.select('id').distinct().count()}")

# Удаляем дубликаты по id
df_raw = df_raw.dropDuplicates(["id"])
print(f"После дедупликации: {df_raw.count()} строк")

# Проверим первые 5 строк
df_raw.select("id", "customer_first_name", "sale_date").show(5)

Всего строк: 20022
Уникальных id: 1072
После дедупликации: 1072 строк
+----+-------------------+----------+
|  id|customer_first_name| sale_date|
+----+-------------------+----------+
|   1|             Barron| 5/14/2021|
|  10|              Jorry| 1/21/2021|
| 100|               Lory|11/28/2021|
|1000|               Seth|10/16/2021|
| 101|             Alexis| 11/3/2021|
+----+-------------------+----------+
only showing top 5 rows



In [3]:
# Удаляем полные дубликаты строк
df_raw = df_raw.distinct()
print(f"После удаления полных дубликатов: {df_raw.count()} строк")

# Приводим типы данных для числовых колонок (Spark прочитал как string)
from pyspark.sql.functions import col, to_date, to_timestamp

# Исправляем проблемные колонки
df_raw = df_raw \
    .withColumn("customer_age", col("customer_age").cast("int")) \
    .withColumn("product_price", col("product_price").cast("decimal(10,2)")) \
    .withColumn("sale_date", to_date(col("sale_date"), "M/d/yyyy")) \
    .withColumn("product_release_date", to_date(col("product_release_date"), "M/d/yyyy")) \
    .withColumn("product_expiry_date", to_date(col("product_expiry_date"), "M/d/yyyy"))

print("Типы исправлены")
df_raw.printSchema()

После удаления полных дубликатов: 1072 строк
Типы исправлены
root
 |-- id: string (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_postal_code: string (nullable = true)
 |-- customer_pet_type: string (nullable = true)
 |-- customer_pet_name: string (nullable = true)
 |-- customer_pet_breed: string (nullable = true)
 |-- seller_first_name: string (nullable = true)
 |-- seller_last_name: string (nullable = true)
 |-- seller_email: string (nullable = true)
 |-- seller_country: string (nullable = true)
 |-- seller_postal_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_price: decimal(10,2) (nullable = true)
 |-- product_quantity: string (nullable = true)
 |-- sale_date: date (nullable 

In [4]:
from pyspark.sql.functions import monotonically_increasing_id

# Перечитываем ВСЕ csv (не удаляя дубли)
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("nullValue", "") \
    .csv("/home/jovyan/mock_data/MOCK_DATA_*.csv")

print(f"Всего строк из 10 файлов: {df_raw.count()}")

# Приводим типы
df_raw = df_raw \
    .withColumn("customer_age", col("customer_age").cast("int")) \
    .withColumn("product_price", col("product_price").cast("decimal(10,2)")) \
    .withColumn("sale_date", to_date(col("sale_date"), "M/d/yyyy")) \
    .withColumn("product_release_date", to_date(col("product_release_date"), "M/d/yyyy")) \
    .withColumn("product_expiry_date", to_date(col("product_expiry_date"), "M/d/yyyy"))

# Генерируем уникальные sale_id от 1 до 10000
df_raw = df_raw.withColumn("sale_id", monotonically_increasing_id() + 1)

# Оставляем ровно 10000 записей
df_raw = df_raw.limit(10000)
print(f"Финальное количество: {df_raw.count()}")
df_raw.select("sale_id", "customer_first_name", "sale_date").show(5)

Всего строк из 10 файлов: 20022
Финальное количество: 10000
+-------+--------------------+----------+
|sale_id| customer_first_name| sale_date|
+-------+--------------------+----------+
|      1|              Barron|2021-05-14|
|      2|                 2.1|      NULL|
|      3|                 Ham|2021-11-13|
|      4|            Farleigh|2021-12-04|
|      5| est et tempus se...|      NULL|
+-------+--------------------+----------+
only showing top 5 rows



In [5]:
# Параметры подключения
pg_url = "jdbc:postgresql://postgres:5432/bigdata_lab"
pg_props = {
    "user": "student",
    "password": "student123",
    "driver": "org.postgresql.Driver"
}

# Записываем raw_mock_data
df_raw.write \
    .jdbc(url=pg_url, table="raw_mock_data", mode="overwrite", properties=pg_props)

print("raw_mock_data записана в PostgreSQL")

# Проверим — прочитаем обратно
df_check = spark.read.jdbc(url=pg_url, table="raw_mock_data", properties=pg_props)
print(f"Проверка: {df_check.count()} строк в raw_mock_data")

raw_mock_data записана в PostgreSQL
Проверка: 10000 строк в raw_mock_data


In [6]:
from pyspark.sql.functions import col, year, quarter, month, dayofmonth, dayofweek, date_format, concat, lit

# Извлекаем уникальные даты из raw_mock_data
df_date = df_raw.select("sale_date").distinct().dropna()
df_date = df_date.withColumnRenamed("sale_date", "full_date")

# Вычисляем атрибуты даты
df_date = df_date \
    .withColumn("year", year(col("full_date"))) \
    .withColumn("quarter", quarter(col("full_date"))) \
    .withColumn("month", month(col("full_date"))) \
    .withColumn("month_name", date_format(col("full_date"), "MMMM")) \
    .withColumn("day", dayofmonth(col("full_date"))) \
    .withColumn("day_of_week", dayofweek(col("full_date"))) \
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))

print(f"Уникальных дат: {df_date.count()}")
df_date.show(5)

# Записываем в PostgreSQL
df_date.write.jdbc(url=pg_url, table="dim_date", mode="overwrite", properties=pg_props)
print("dim_date записана")

Уникальных дат: 364
+----------+----+-------+-----+----------+---+-----------+--------+
| full_date|year|quarter|month|month_name|day|day_of_week|day_name|
+----------+----+-------+-----+----------+---+-----------+--------+
|2021-05-14|2021|      2|    5|       May| 14|          6|  Friday|
|2021-11-13|2021|      4|   11|  November| 13|          7|Saturday|
|2021-12-04|2021|      4|   12|  December|  4|          7|Saturday|
|2021-08-10|2021|      3|    8|    August| 10|          3| Tuesday|
|2021-02-04|2021|      1|    2|  February|  4|          5|Thursday|
+----------+----+-------+-----+----------+---+-----------+--------+
only showing top 5 rows

dim_date записана


In [7]:
# Уникальные комбинации питомцев
df_pet = df_raw.select("customer_pet_type", "customer_pet_name", "customer_pet_breed") \
    .distinct() \
    .dropna()

df_pet = df_pet \
    .withColumnRenamed("customer_pet_type", "pet_type") \
    .withColumnRenamed("customer_pet_name", "pet_name") \
    .withColumnRenamed("customer_pet_breed", "pet_breed")

print(f"Уникальных питомцев: {df_pet.count()}")
df_pet.show(5)

df_pet.write.jdbc(url=pg_url, table="dim_pet", mode="overwrite", properties=pg_props)
print("dim_pet записана")

Уникальных питомцев: 8126
+--------------------+------------+------------------+
|            pet_type|    pet_name|         pet_breed|
+--------------------+------------+------------------+
|                 cat|   Priscella|Labrador Retriever|
|bmassingham0@unbl...|914-877-7062|          Suite 25|
|                bird|     Dalenna|Labrador Retriever|
|                bird|    Aldridge|          Parakeet|
|  vhuxter2@slate.com|434-817-1275|            Apt 96|
+--------------------+------------+------------------+
only showing top 5 rows

dim_pet записана


In [8]:
import pandas as pd

df_check = pd.read_csv("/home/jovyan/mock_data/MOCK_DATA_1.csv")
print("Колонки в CSV (первые 25):")
for i, col_name in enumerate(df_check.columns[:25]):
    sample_val = str(df_check[col_name].iloc[0])
    print(f"  [{i}] {col_name}: {sample_val}")

print("\nСтрока 1, колонки 9-14 (питомец):")
for i in range(9, 15):
    print(f"  [{i}] {df_check.columns[i]}: {df_check.iloc[0, i]}")

Колонки в CSV (первые 25):
  [0] id: 1
  [1] customer_first_name: Conni
  [2] customer_last_name: Leydon
  [3] customer_age: 63
  [4] customer_email: lswait0@amazon.com
  [5] customer_country: France
  [6] customer_postal_code: 77404 CEDEX
  [7] customer_pet_type: cat
  [8] customer_pet_name: Jan
  [9] customer_pet_breed: Labrador Retriever
  [10] seller_first_name: Lenee
  [11] seller_last_name: Swait
  [12] seller_email: lswait0@51.la
  [13] seller_country: Central African Republic
  [14] seller_postal_code: nan
  [15] product_name: Bird Cage
  [16] product_category: Food
  [17] product_price: 18.57
  [18] product_quantity: 87
  [19] sale_date: 2/27/2021
  [20] sale_customer_id: 1
  [21] sale_seller_id: 1
  [22] sale_product_id: 1
  [23] sale_quantity: 3
  [24] sale_total_price: 83.32

Строка 1, колонки 9-14 (питомец):
  [9] customer_pet_breed: Labrador Retriever
  [10] seller_first_name: Lenee
  [11] seller_last_name: Swait
  [12] seller_email: lswait0@51.la
  [13] seller_country: C

In [9]:
import pandas as pd
import os

all_dfs = []
data_dir = "/home/jovyan/mock_data"
files = sorted([f for f in os.listdir(data_dir) if f.startswith("MOCK_DATA_") and f.endswith(".csv")])

for f in files:
    path = os.path.join(data_dir, f)
    df_pd = pd.read_csv(path)
    all_dfs.append(df_pd)

df_all = pd.concat(all_dfs, ignore_index=True)
print(f"Pandas: {len(df_all)} строк")

# Конвертируем в Spark
df_raw = spark.createDataFrame(df_all)

# Приводим типы
from pyspark.sql.functions import col, to_date
df_raw = df_raw \
    .withColumn("product_price", col("product_price").cast("decimal(10,2)")) \
    .withColumn("product_quantity", col("product_quantity").cast("integer")) \
    .withColumn("sale_date", to_date(col("sale_date"), "M/d/yyyy")) \
    .withColumn("product_release_date", to_date(col("product_release_date"), "M/d/yyyy")) \
    .withColumn("product_expiry_date", to_date(col("product_expiry_date"), "M/d/yyyy"))

# Генерируем sale_id
from pyspark.sql.functions import monotonically_increasing_id
df_raw = df_raw.withColumn("sale_id", monotonically_increasing_id() + 1)
df_raw = df_raw.limit(10000)

print(f"Spark: {df_raw.count()} строк")
df_raw.select("customer_pet_type", "customer_pet_name", "customer_pet_breed").show(5)

Pandas: 10000 строк
Spark: 10000 строк
+-----------------+-----------------+------------------+
|customer_pet_type|customer_pet_name|customer_pet_breed|
+-----------------+-----------------+------------------+
|              cat|              Jan|Labrador Retriever|
|              dog|           Shelia|Labrador Retriever|
|              cat|           Gunner|           Siamese|
|             bird|            Nahum|Labrador Retriever|
|              cat|             Brod|Labrador Retriever|
+-----------------+-----------------+------------------+
only showing top 5 rows



In [10]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# === dim_date: с суррогатным ключом ===
window_date = Window.orderBy("full_date")
df_date = df_date.withColumn("date_key", row_number().over(window_date))
df_date.write.jdbc(url=pg_url, table="dim_date", mode="overwrite", properties=pg_props)
print(f"dim_date: {df_date.count()} строк")

# === dim_pet: с суррогатным ключом ===
window_pet = Window.orderBy("pet_type", "pet_name", "pet_breed")
df_pet = df_pet.withColumn("pet_key", row_number().over(window_pet))
df_pet.write.jdbc(url=pg_url, table="dim_pet", mode="overwrite", properties=pg_props)
print(f"dim_pet: {df_pet.count()} строк")

# === dim_customer: уникальные customer_id + связь с pet_key ===
df_customer_unique = df_raw.select(
    "sale_customer_id", "customer_first_name", "customer_last_name",
    "customer_age", "customer_email", "customer_country", "customer_postal_code",
    "customer_pet_type", "customer_pet_name", "customer_pet_breed"
).distinct().dropna(subset=["sale_customer_id"])

# Джойним с dim_pet чтобы получить pet_key
df_customer_with_pet = df_customer_unique.join(
    df_pet,
    (df_customer_unique.customer_pet_type == df_pet.pet_type) &
    (df_customer_unique.customer_pet_name == df_pet.pet_name) &
    (df_customer_unique.customer_pet_breed == df_pet.pet_breed),
    "left"
).select(
    col("sale_customer_id").alias("customer_id"),
    col("customer_first_name").alias("first_name"),
    col("customer_last_name").alias("last_name"),
    "customer_age",
    "customer_email",
    "customer_country",
    "customer_postal_code",
    "pet_key"
)

window_cust = Window.orderBy("customer_id")
df_customer_final = df_customer_with_pet.withColumn("customer_key", row_number().over(window_cust))
df_customer_final.write.jdbc(url=pg_url, table="dim_customer", mode="overwrite", properties=pg_props)
print(f"dim_customer: {df_customer_final.count()} строк (с pet_key)")

# === dim_seller: с суррогатным ключом ===
df_seller_unique = df_raw.select(
    "sale_seller_id", "seller_first_name", "seller_last_name",
    "seller_email", "seller_country", "seller_postal_code"
).distinct().dropna(subset=["sale_seller_id"]) \
 .withColumnRenamed("sale_seller_id", "seller_id") \
 .withColumnRenamed("seller_first_name", "first_name") \
 .withColumnRenamed("seller_last_name", "last_name")

window_seller = Window.orderBy("seller_id")
df_seller_final = df_seller_unique.withColumn("seller_key", row_number().over(window_seller))
df_seller_final.write.jdbc(url=pg_url, table="dim_seller", mode="overwrite", properties=pg_props)
print(f"dim_seller: {df_seller_final.count()} строк")

# === dim_supplier: с суррогатным ключом ===
df_supplier_unique = df_raw.select(
    "supplier_name", "supplier_contact", "supplier_email",
    "supplier_phone", "supplier_address", "supplier_city", "supplier_country"
).distinct().dropna(subset=["supplier_name"])

window_supplier = Window.orderBy("supplier_name")
df_supplier_final = df_supplier_unique.withColumn("supplier_key", row_number().over(window_supplier))
df_supplier_final.write.jdbc(url=pg_url, table="dim_supplier", mode="overwrite", properties=pg_props)
print(f"dim_supplier: {df_supplier_final.count()} строк")

# === dim_product: с supplier_key ===
df_product_unique = df_raw.select(
    "sale_product_id", "product_name", "product_category", "product_price",
    "product_weight", "product_color", "product_size", "product_brand",
    "product_material", "product_description", "product_rating", "product_reviews",
    "product_release_date", "product_expiry_date", "pet_category", "supplier_name"
).distinct().dropna(subset=["sale_product_id"])

df_product_with_supplier = df_product_unique.join(
    df_supplier_final.select("supplier_name", "supplier_key"),
    "supplier_name",
    "left"
).select(
    col("sale_product_id").alias("product_id"),
    "product_name", "product_category", "product_price",
    "product_weight", "product_color", "product_size", "product_brand",
    "product_material", "product_description", "product_rating", "product_reviews",
    "product_release_date", "product_expiry_date", "pet_category", "supplier_key"
)

window_prod = Window.orderBy("product_id")
df_product_final = df_product_with_supplier.withColumn("product_key", row_number().over(window_prod))
df_product_final.write.jdbc(url=pg_url, table="dim_product", mode="overwrite", properties=pg_props)
print(f"dim_product: {df_product_final.count()} строк (с supplier_key)")

# === dim_store: с суррогатным ключом ===
df_store_unique = df_raw.select(
    "store_name", "store_location", "store_city", "store_state",
    "store_country", "store_phone", "store_email"
).distinct().dropna(subset=["store_name"])

window_store = Window.orderBy("store_name")
df_store_final = df_store_unique.withColumn("store_key", row_number().over(window_store))
df_store_final.write.jdbc(url=pg_url, table="dim_store", mode="overwrite", properties=pg_props)
print(f"dim_store: {df_store_final.count()} строк")

print("\nВсе измерения со связями созданы")

dim_date: 364 строк
dim_pet: 8126 строк
dim_customer: 10000 строк (с pet_key)
dim_seller: 10000 строк
dim_supplier: 10000 строк
dim_product: 281170 строк (с supplier_key)
dim_store: 10000 строк

Все измерения со связями созданы


In [11]:
# Проверим, сколько уникальных product_id
df_prod_check = df_raw.select("sale_product_id").distinct().dropna()
print(f"Уникальных product_id: {df_prod_check.count()}")

# Возьмём уникальные продукты до джойна
df_product_unique = df_raw.select(
    "sale_product_id", "product_name", "product_category", "product_price",
    "product_weight", "product_color", "product_size", "product_brand",
    "product_material", "product_description", "product_rating", "product_reviews",
    "product_release_date", "product_expiry_date", "pet_category", "supplier_name"
).dropna(subset=["sale_product_id"]).dropDuplicates(["sale_product_id"])

print(f"Уникальных продуктов после dropDuplicates: {df_product_unique.count()}")

# Джойн с dim_supplier
df_product_with_supplier = df_product_unique.join(
    df_supplier_final.select("supplier_name", "supplier_key"),
    "supplier_name",
    "left"
).select(
    col("sale_product_id").alias("product_id"),
    "product_name", "product_category", "product_price",
    "product_weight", "product_color", "product_size", "product_brand",
    "product_material", "product_description", "product_rating", "product_reviews",
    "product_release_date", "product_expiry_date", "pet_category", "supplier_key"
)

window_prod = Window.orderBy("product_id")
df_product_final = df_product_with_supplier.withColumn("product_key", row_number().over(window_prod))

df_product_final.write.jdbc(url=pg_url, table="dim_product", mode="overwrite", properties=pg_props)
print(f"dim_product: {df_product_final.count()} строк (с supplier_key)")

Уникальных product_id: 1000
Уникальных продуктов после dropDuplicates: 1000
dim_product: 27957 строк (с supplier_key)


In [12]:
# Берём уникальные продукты
df_product_unique = df_raw.select(
    "sale_product_id", "product_name", "product_category", "product_price",
    "product_weight", "product_color", "product_size", "product_brand",
    "product_material", "product_description", "product_rating", "product_reviews",
    "product_release_date", "product_expiry_date", "pet_category", "supplier_name"
).dropna(subset=["sale_product_id"]).dropDuplicates(["sale_product_id"])

# Получаем supplier_key через один supplier_name на supplier_key
df_supplier_mapping = df_supplier_final.select("supplier_key", "supplier_name").dropDuplicates(["supplier_name"])

# Джойним
df_product_with_supplier = df_product_unique.join(
    df_supplier_mapping,
    "supplier_name",
    "left"
)

# Убираем supplier_name и оставляем supplier_key
df_product_final = df_product_with_supplier.select(
    col("sale_product_id").alias("product_id"),
    "product_name", "product_category", "product_price",
    "product_weight", "product_color", "product_size", "product_brand",
    "product_material", "product_description", "product_rating", "product_reviews",
    "product_release_date", "product_expiry_date", "pet_category", "supplier_key"
).dropDuplicates(["product_id"])

window_prod = Window.orderBy("product_id")
df_product_final = df_product_final.withColumn("product_key", row_number().over(window_prod))

print(f"dim_product: {df_product_final.count()} строк")
df_product_final.write.jdbc(url=pg_url, table="dim_product", mode="overwrite", properties=pg_props)
print("dim_product записана")

dim_product: 1000 строк
dim_product записана


In [13]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# === dim_date (уже ок) ===
window_date = Window.orderBy("full_date")
df_date = df_date.dropDuplicates(["full_date"])
df_date = df_date.withColumn("date_key", row_number().over(window_date))
df_date.write.jdbc(url=pg_url, table="dim_date", mode="overwrite", properties=pg_props)
print(f"dim_date: {df_date.count()}")

# === dim_pet ===
window_pet = Window.orderBy("pet_type", "pet_name", "pet_breed")
df_pet = df_pet.dropDuplicates(["pet_type", "pet_name", "pet_breed"])
df_pet = df_pet.withColumn("pet_key", row_number().over(window_pet))
df_pet.write.jdbc(url=pg_url, table="dim_pet", mode="overwrite", properties=pg_props)
print(f"dim_pet: {df_pet.count()}")

# === dim_customer (уникальные customer_id) ===
window_cust = Window.orderBy("customer_id")
df_customer_final = df_customer_final.dropDuplicates(["customer_id"])
df_customer_final = df_customer_final.withColumn("customer_key", row_number().over(window_cust))
df_customer_final.write.jdbc(url=pg_url, table="dim_customer", mode="overwrite", properties=pg_props)
print(f"dim_customer: {df_customer_final.count()}")

# === dim_seller (уникальные seller_id) ===
window_seller = Window.orderBy("seller_id")
df_seller_final = df_seller_final.dropDuplicates(["seller_id"])
df_seller_final = df_seller_final.withColumn("seller_key", row_number().over(window_seller))
df_seller_final.write.jdbc(url=pg_url, table="dim_seller", mode="overwrite", properties=pg_props)
print(f"dim_seller: {df_seller_final.count()}")

# === dim_product (уникальные product_id) ===
window_prod = Window.orderBy("product_id")
df_product_final = df_product_final.dropDuplicates(["product_id"])
df_product_final = df_product_final.withColumn("product_key", row_number().over(window_prod))
df_product_final.write.jdbc(url=pg_url, table="dim_product", mode="overwrite", properties=pg_props)
print(f"dim_product: {df_product_final.count()}")

# === dim_supplier (уникальные supplier_name) ===
window_supp = Window.orderBy("supplier_name")
df_supplier_final = df_supplier_final.dropDuplicates(["supplier_name"])
df_supplier_final = df_supplier_final.withColumn("supplier_key", row_number().over(window_supp))
df_supplier_final.write.jdbc(url=pg_url, table="dim_supplier", mode="overwrite", properties=pg_props)
print(f"dim_supplier: {df_supplier_final.count()}")

# === dim_store (уникальные store_name) ===
window_store = Window.orderBy("store_name")
df_store_final = df_store_final.dropDuplicates(["store_name"])
df_store_final = df_store_final.withColumn("store_key", row_number().over(window_store))
df_store_final.write.jdbc(url=pg_url, table="dim_store", mode="overwrite", properties=pg_props)
print(f"dim_store: {df_store_final.count()}")

print("\nВсе измерения исправлены")

dim_date: 364
dim_pet: 8126
dim_customer: 1000
dim_seller: 1000
dim_product: 1000
dim_supplier: 383
dim_store: 383

Все измерения исправлены


In [14]:
from pyspark.sql.functions import sum as spark_sum

# Готовим факты
df_fact = df_raw.select(
    "sale_id", "sale_date", "sale_customer_id", "sale_seller_id",
    "sale_product_id", "sale_quantity", "sale_total_price",
    "store_name"
)

# Джойн с dim_date (df_date содержит date_key и full_date)
df_fact = df_fact.join(
    df_date.select("date_key", "full_date"),
    df_fact.sale_date == df_date.full_date,
    "left"
)

# Джойн с dim_customer
df_fact = df_fact.join(
    df_customer_final.select("customer_key", "customer_id"),
    df_fact.sale_customer_id == df_customer_final.customer_id,
    "left"
)

# Джойн с dim_seller
df_fact = df_fact.join(
    df_seller_final.select("seller_key", "seller_id"),
    df_fact.sale_seller_id == df_seller_final.seller_id,
    "left"
)

# Джойн с dim_product
df_fact = df_fact.join(
    df_product_final.select("product_key", "product_id"),
    df_fact.sale_product_id == df_product_final.product_id,
    "left"
)

# Джойн с dim_store
df_fact = df_fact.join(
    df_store_final.select("store_key", "store_name"),
    "store_name",
    "left"
)

# Оставляем только нужные колонки
df_fact_final = df_fact.select(
    "sale_id", "date_key", "customer_key", "seller_key",
    "product_key", "store_key", "sale_quantity", "sale_total_price"
)

print(f"fact_sale: {df_fact_final.count()} строк")

# Проверка целостности связей
df_fact_final.select(
    spark_sum(col("date_key").isNull().cast("int")).alias("null_date"),
    spark_sum(col("customer_key").isNull().cast("int")).alias("null_customer"),
    spark_sum(col("seller_key").isNull().cast("int")).alias("null_seller"),
    spark_sum(col("product_key").isNull().cast("int")).alias("null_product"),
    spark_sum(col("store_key").isNull().cast("int")).alias("null_store")
).show()

# Записываем
df_fact_final.write.jdbc(url=pg_url, table="fact_sale", mode="overwrite", properties=pg_props)
print("fact_sale записана")

fact_sale: 10000 строк
+---------+-------------+-----------+------------+----------+
|null_date|null_customer|null_seller|null_product|null_store|
+---------+-------------+-----------+------------+----------+
|        0|            0|          0|           0|         0|
+---------+-------------+-----------+------------+----------+

fact_sale записана


In [15]:
# Проверим все таблицы
tables = ["raw_mock_data", "dim_date", "dim_pet", "dim_customer", 
          "dim_seller", "dim_product", "dim_store", "dim_supplier", "fact_sale"]

for t in tables:
    df_check = spark.read.jdbc(url=pg_url, table=t, properties=pg_props)
    print(f"{t}: {df_check.count()} строк, {len(df_check.columns)} колонок")

raw_mock_data: 10000 строк, 51 колонок
dim_date: 364 строк, 9 колонок
dim_pet: 8126 строк, 4 колонок
dim_customer: 1000 строк, 9 колонок
dim_seller: 1000 строк, 7 колонок
dim_product: 1000 строк, 17 колонок
dim_store: 383 строк, 8 колонок
dim_supplier: 383 строк, 8 колонок
fact_sale: 10000 строк, 8 колонок
